# Task 4.8: Reentrenamiento final de SegFormer y UWSegFormer

Comparación con U-Net. Usa ground truth reconciliado,
un manifiesto LOVO común, 3–5 semillas y checkpoint final sin selección sobre test.
Exporta mIoU y Dice (media ± desviación entre semillas) y resultados por frame.
Las celdas exigen imágenes, máscaras, pesos y hash reales.
La configuración inicial bloquea el entrenamiento hasta fijar el presupuesto y el manifiesto.

## 1. Cargar las fuentes locales

Desde el repositorio: `python scripts/package_final_retraining.py`.
Subir el ZIP resultante en la celda siguiente.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

SOURCE_ROOT = Path('/content/task4_8_sources')
archive_path = Path('/content/task4_8_sources.zip')
if not archive_path.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Sube únicamente task4_8_sources.zip')
    archive_path.write_bytes(next(iter(uploaded.values())))
with zipfile.ZipFile(archive_path) as archive:
    for item in archive.infolist():
        target = (SOURCE_ROOT / item.filename).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()):
            raise ValueError('Ruta inválida en ZIP')
    archive.extractall(SOURCE_ROOT)
print(SOURCE_ROOT)


## 2. Entorno real de entrenamiento

Activa GPU en Colab. La primera celda instala CondaColab y reinicia el runtime si es necesario.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec('condacolab') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'], check=True)
import condacolab
condacolab.install()


In [ ]:
from pathlib import Path
import subprocess

SOURCE_ROOT = Path('/content/task4_8_sources')
if not (SOURCE_ROOT / 'src/training/final_retraining.py').exists():
    raise FileNotFoundError('Ejecuta primero la carga del ZIP de fuentes')
existing = subprocess.check_output(['conda', 'env', 'list'], text=True)
if not any(line.split() and line.split()[0] == 'uws' for line in existing.splitlines()):
    subprocess.run(['conda', 'create', '-n', 'uws', 'python=3.8', '-y'], check=True)
    subprocess.run(['conda', 'run', '-n', 'uws', 'python', '-m', 'pip', 'install',
        'torch==1.10.2+cu111', 'torchvision==0.11.3+cu111',
        '-f', 'https://download.pytorch.org/whl/torch_stable.html'], check=True)
    subprocess.run(['conda', 'run', '-n', 'uws', 'python', '-m', 'pip', 'install',
        'mmcv-full==1.5.3', '-f', 'https://download.openmmlab.com/mmcv/dist/cu111/torch1.10/index.html'], check=True)
subprocess.run(['conda', 'run', '-n', 'uws', 'python', '-m', 'pip', 'install',
                '-r', str(SOURCE_ROOT / 'requirements-final-retraining.txt')], check=True)


## 3. Código y pesos preentrenados reales

Se conserva el stack y el parche de imports de los notebooks anteriores.
SegFormer usa el encoder ADE20K; UWSegFormer usa tu checkpoint SUIM de Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import urllib.request

UWS_ROOT = Path('/content/UWSegFormer')
UWS_COMMIT = 'b61fa450bad58137b3ab8fecd1e7f7cedba4ce57'
if not UWS_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SAWRJJ/UWSegFormer.git', str(UWS_ROOT)], check=True)
actual = subprocess.check_output(['git', '-C', str(UWS_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if actual != UWS_COMMIT:
    subprocess.run(['git', '-C', str(UWS_ROOT), 'checkout', UWS_COMMIT], check=True)
init_path = UWS_ROOT / 'mmseg/models/decode_heads/__init__.py'
text = init_path.read_text()
text = text.replace('from .segformer_head import PPMobileSegHead_segformer2,PPMobileSegHead_segformer1',
                    'from .segformer_head import MAA_segformer1')
text = text.replace("    'PPMobileSegHead_segformer2',\n    'PPMobileSegHead_segformer1'", "    'MAA_segformer1'")
init_path.write_text(text)

SEG_CHECKPOINT = Path('/content/segformer_mit_b0_ade20k.pth')
SEG_URL = ('https://download.openmmlab.com/mmsegmentation/v0.5/segformer/'
           'segformer_mit-b0_512x512_160k_ade20k/'
           'segformer_mit-b0_512x512_160k_ade20k_20210726_101530-8ffa8fda.pth')
if not SEG_CHECKPOINT.exists():
    temporary = SEG_CHECKPOINT.with_suffix('.download')
    urllib.request.urlretrieve(SEG_URL, temporary)
    temporary.replace(SEG_CHECKPOINT)

runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(SOURCE_ROOT) + ':' + str(UWS_ROOT)
runtime_env['LD_LIBRARY_PATH'] = '/usr/local/envs/uws/lib:' + runtime_env.get('LD_LIBRARY_PATH', '')
runtime_env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
subprocess.run(['conda', 'run', '--no-capture-output', '-n', 'uws', 'python', '-c',
    "import sys, torch, mmcv; from mmseg.models.builder import HEADS; "
    "assert sys.version_info[:2] == (3,8); assert torch.__version__.startswith('1.10.2'); "
    "assert mmcv.__version__ == '1.5.3'; assert torch.cuda.is_available(); "
    "assert HEADS.get('SegFormerHead') and HEADS.get('MAA_segformer1'); "
    "print(torch.cuda.get_device_name(0))"], cwd=SOURCE_ROOT, env=runtime_env, check=True)


## 4. Configuración acordada

Completa `manifest_sha256`, `expected_videos` y `passes`. Usa las máscaras reconciliadas;
no hay fallback a SAM. El manifiesto debe tener un fold por video incluido.

Para el ensayo técnico: `mode='pilot'`, una semilla, un fold y un presupuesto fijado de antemano.
No usar las métricas del video de prueba para elegir duración o hiperparámetros.

Para el final: `mode='final'`, `protocol_confirmed=True`, `seeds=[0,1,2,3,4]`, `folds=None`.
Los cinco entrenamientos de cada fold vuelven a inicializar desde el checkpoint preentrenado.

In [ ]:
import json
settings = json.loads((SOURCE_ROOT / 'configs/final_retraining_segformers.example.json').read_text())
# EDITAR
settings['manifest_sha256'] = None
settings['passes'] = None
settings['mode'] = 'pilot'
settings['protocol_confirmed'] = False
settings['seeds'] = [0]
settings['folds'] = ['Video1']
# Confirmar la lista real; no reducirla automáticamente si faltan etiquetas.
settings['expected_videos'] = ['Video1', 'Video2', 'Video3', 'Video4', 'Video5', 'Video6']
settings_path = SOURCE_ROOT / 'settings.json'
settings_path.write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))


### Preparar o verificar el manifiesto real de 64 máscaras

Acuerdo: 24 de Video1 y 8 de cada video 2–6; Video7 queda fuera. Esta celda verifica los
conteos y todos los pares reales. Si el CSV compartido ya existe, lo verifica sin modificarlo.
Si no existe, lo crea para compartirlo. Conserva el SHA-256 impreso y
confirma que se usara **ese mismo archivo** antes de copiarlo a `manifest_sha256` arriba.
Luego vuelve a ejecutar la celda de configuración. No inventes el hash ni el presupuesto.


In [ ]:
manifest_command = ['conda', 'run', '--no-capture-output', '-n', 'uws', 'python',
    '-m', 'src.data.build_final_retraining_manifest',
    '--frames-dir', settings['frames_dir'], '--masks-dir', settings['masks_dir'],
    '--out', settings['manifest']]
subprocess.run(manifest_command, cwd=SOURCE_ROOT, env=runtime_env, check=True)


## 5. Preflight (no entrena)

Revisa todos los pares, etiquetas, dimensiones y splits, y muestra las iteraciones por modelo,
fold y semilla. Si faltan acuerdos o datos, se detiene.


In [ ]:
command = ['conda', 'run', '--no-capture-output', '-n', 'uws', 'python',
           '-m', 'src.training.final_retraining', '--settings', str(settings_path)]
subprocess.run(command, cwd=SOURCE_ROOT, env=runtime_env, check=True)


## 6. Entrenamiento e inferencia reales

Activa `RUN` solo tras revisar el preflight. El test no participa en selección de checkpoints.
Cada corrida guarda sus pesos finales y predicciones. Ante interrupción conserva/mueve la carpeta
parcial antes de reintentar. No se reanudan pesos de otro fold o semilla.


In [ ]:
RUN = False
if RUN:
    subprocess.run(command + ['--run'], cwd=SOURCE_ROOT, env=runtime_env, check=True)
else:
    print('Sin ejecutar entrenamientos. Activa RUN tras revisar el plan.')


## 7. Inspección visual de una predicción real

Ejecutar después de una corrida terminada.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
result_root = Path(settings['results_dir']) / settings['mode']
predictions = sorted(result_root.glob('*/predictions/*.png'))
if not predictions:
    raise FileNotFoundError('Todavía no hay predicciones reales')
prediction = predictions[0]
dataset = prediction.parent.parent / 'dataset'
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, path, title in zip(axes,
    [dataset / 'images/test' / prediction.name,
     dataset / 'annotations/test' / prediction.name, prediction],
    ['Frame', 'Ground truth reconciliado', 'Predicción final']):
    ax.imshow(Image.open(path)); ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.show()


## 8. Tabla final conjunta, requiere resultados completos de los tres modelos

Las corridas finales ya exportan los `per_frame` esperados por Ariana y una tabla de los dos
SegFormers. La tabla conjunta se genera solo al completar los tres modelos y todas las semillas.

Ariana debe añadir `miou` y `mdice` a sus exports usando `score_predictions(preds,gts)` del módulo
compartido. `iou` y `dice` se conservan como métricas de estructura. mIoU promedia fondo/estructura
por imagen; no es la mIoU acumulada de MMSeg. Ver `docs/final_retraining_segformers.md`.


In [ ]:
BUILD_FINAL_TABLE = False
if BUILD_FINAL_TABLE:
    subprocess.run(command + ['--table'], cwd=SOURCE_ROOT, env=runtime_env, check=True)
else:
    print('Tabla conjunta pendiente de los resultados completos de U-Net, SegFormer y UWSegFormer.')
